# Collect Users and Group Memberships from MS Graph, save to Lakehouse table

## Config

In [ ]:
KEYVAULT = 'https://<<REDACTED>>.vault.azure.net/'

#service principal will need application grant of Directory.ReadAll
CLIENT_ID = notebookutils.credentials.getSecret(KEYVAULT, "<<REDACTED>>")
CLIENT_SECRET = notebookutils.credentials.getSecret(KEYVAULT, '<<REDACTED>>')
TENANT_ID = notebookutils.credentials.getSecret(KEYVAULT, '<<REDACTED>>')

LAKEHOUSE_WORKSPACE_ID="<<REDACTED>>"
LAKEHOUSE_GUID="<<REDACTED>>"
LAKEHOUSE_URL = f"abfss://{LAKEHOUSE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_GUID}"

USER_DEST_TABLE_SCHEMA="it"
USER_DEST_TABLE_NAME="users_bronze"

GROUP_DEST_TABLE_SCHEMA="it"
GROUP_DEST_TABLE_NAME="groups_bronze"

MEMBERSHIP_DEST_TABLE_SCHEMA="it"
MEMBERSHIP_DEST_TABLE_NAME="group_members_bronze"

SNAPSHOT_BASE = "Files/it/bronze"



## Setup work, authenticate

### imports and function defs

In [ ]:
import requests
import logging
from datetime import datetime
from msal import ConfidentialClientApplication
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType

#config logging
FORMAT = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
formatter = logging.Formatter(fmt=FORMAT)
for handler in logging.getLogger().handlers:
    handler.setFormatter(formatter)
logging.getLogger().setLevel(logging.WARN)
logger = logging.getLogger('default')

In [ ]:
def fetch_graph_items(uri, access_token, page_limit=None):
    """
    Fetch all pages of data from Microsoft Graph API with proper pagination handling.
    
    Args:
        uri (str): The API endpoint to query
        access_token (str): Welp, it's an access token. you know you need one, not sure what to add. 
        page_limit (int, optional): Maximum number of pages to fetch. Defaults to None (fetch all).
        
    Returns:
        list: All items retrieved from the API
    """       
    headers = {"Authorization": f"Bearer {access_token}"}
    all_items = []
    next_link = uri
    page = 1
    
    try:
        while next_link:
            if page % 10 == 0:
                logger.info(f"Fetching page {page}...")
                
            response = requests.get(next_link, headers=headers)
            response.raise_for_status()
            
            data = response.json()
            items = data.get("value", [])
            all_items.extend(items)
            
            # Check if we've reached the page limit
            if page_limit and page >= page_limit:
                logger.info(f"Reached page limit of {page_limit}, stopping pagination")
                break
                
            # Get the next page URL if it exists
            next_link = data.get("@odata.nextLink")
            page += 1
            
        logger.info(f'Successfully fetched {len(all_items)} items from {page-1} pages')
        return all_items
        
    except requests.exceptions.RequestException as e:
        logger.error(f"Request failed: {str(e)}")
        if hasattr(e, 'response') and e.response:
            try:
                error_details = e.response.json()
                logger.error(f"Error details: {error_details}")
            except ValueError:
                logger.error(f"Error response (not JSON): {e.response.text}")
        #reraise error
        raise        
        

In [ ]:
def save_to_lakehouse_with_snapshot(df, schema_name, table_name):
    # Save to lakehouse
    delta_table_path = LAKEHOUSE_URL+f"/Tables/{schema_name}/{table_name}"
    logger.info(f'saving to {delta_table_path}')
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_table_path)

    #save a snapshot on sundays    
    if datetime.now().weekday() == 6:
        snapshot_path = f'{SNAPSHOT_BASE}/{datetime.now().strftime("%Y%m%d-%H%M")}/{table_name.lower()}.csv'
        df.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(snapshot_path)

### login to graph and get a magic token

In [ ]:
# Create a confidential client application
logger.info('Fetching secrets from vault')
app = ConfidentialClientApplication(
    CLIENT_ID,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    client_credential=CLIENT_SECRET,
)

# Acquire a token
token_response = app.acquire_token_for_client(
    scopes=["https://graph.microsoft.com/.default"]
)

if "access_token" not in token_response:
    logger.error("Error acquiring token")
    logger.error(token_response.get("error"))
    logger.error(token_response.get("error_description"))
    raise('Unable to get access token from login')
access_token = token_response["access_token"]

## Users

In [ ]:
GRAPH_USER_QUERY = "https://graph.microsoft.com/v1.0/users?$select=id,accountEnabled,displayName,givenName,surname,employeeType,createdDateTime,department,jobTitle,mail,userPrincipalName&$expand=manager($select=id,displayName)"
all_users = fetch_graph_items(GRAPH_USER_QUERY, access_token)

# Create DataFrame
df = spark.createDataFrame(all_users)

# Define schema for the manager column
manager_schema = StructType([
    StructField("id", StringType(), True),
    StructField("displayName", StringType(), True)
])

# Expand the manager column into manager_id and manager_displayName
df = df.withColumn("manager_json", from_json(col("manager").cast("string"), manager_schema))
df = df.withColumn("manager_id", col("manager_json.id"))
df = df.withColumn("manager_displayName", col("manager_json.displayName"))

# Drop the original manager column
df = df.drop("manager").drop("manager_json")

# Convert createdDateTime to timestamp
df = df.withColumn("createdDateTime", to_timestamp(col("createdDateTime")))

# Save to lakehouse
save_to_lakehouse_with_snapshot(df, USER_DEST_TABLE_SCHEMA, USER_DEST_TABLE_NAME)

## Groups and Group Memberships

In [ ]:
GRAPH_GROUP_QUERY = ("https://graph.microsoft.com/v1.0/groups?"
    "$select=id,displayName,description,visibility,createdDateTime,mail,groupTypes,mailEnabled,securityEnabled"
    #Edit filter to your liking
    "&$filter=startswith(displayName, 'Report') or "
             "startswith(displayName, '%23Report')")
group_list = fetch_graph_items(GRAPH_GROUP_QUERY, access_token)

#collapse single member list
#see https://learn.microsoft.com/en-us/graph/api/resources/groups-overview?view=graph-rest-1.0&tabs=http#group-types-in-microsoft-entra-id-and-microsoft-graph
for g in group_list:
    g['groupTypes'] = ','.join(g['groupTypes'])

# Create DataFrame
df = spark.createDataFrame(group_list)

# Convert createdDateTime to timestamp
df = df.withColumn("createdDateTime", to_timestamp(col("createdDateTime")))

# Save to lakehouse
save_to_lakehouse_with_snapshot(df, GROUP_DEST_TABLE_SCHEMA, GROUP_DEST_TABLE_NAME)

In [ ]:
group_memberships = []

for group in group_list:
    uri = f"https://graph.microsoft.com/v1.0/groups/{group['id']}/members?$select=id"
    users = fetch_graph_items(uri, access_token)
    group_memberships.extend({"group_id": group['id'], "user_id": user["id"]} for user in users)

# Save to lakehouse
df = spark.createDataFrame(group_memberships)
save_to_lakehouse_with_snapshot(df, MEMBERSHIP_DEST_TABLE_SCHEMA, MEMBERSHIP_DEST_TABLE_NAME)